**Text Generation with Recurrent Neural Networks (RNNs)**

In [1]:
textpath = r"D:\DeepLearning\DataSet\Text\anna.txt"

with open(textpath) as f:
    text=f.read()
words=text.split(" ") 

print(words[:20])
print(set(text.lower()))

['Chapter', '1\n\n\n\n\n\nHappy', 'families', 'are', 'all', 'alike;', 'every', 'unhappy', 'family', 'is', 'unhappy', 'in', 'its', 'own\n\nway.\n\n\n\nEverything', 'was', 'in', 'confusion', 'in', 'the', "Oblonskys'"]
{'t', 'v', 'o', 'w', '!', 'x', 'l', 'b', 'd', '(', 'e', "'", '?', 'n', 'q', '7', 'j', ')', '4', '`', 'm', 'i', 'z', 'g', '8', '6', '1', '.', 'h', 'r', '"', 's', '2', 'k', '*', 'p', '9', ';', 'u', '\n', '-', 'y', 'f', ':', 'c', '_', '5', ' ', '0', '3', 'a', ','}


In [ ]:
clean_text=text.lower().replace("\n", " ")
clean_text=clean_text.replace("-", " ")
for x in ",.:;?!$()/_&%*@'`":
    clean_text=clean_text.replace(f"{x}", f" {x} ")
clean_text=clean_text.replace('"', ' " ') 
text=clean_text.split()

In [3]:
from collections import Counter   
word_counts = Counter(text)    

# get unique words
words=sorted(word_counts, key=word_counts.get,
                      reverse=True) 
print(words[:10])

[',', '.', 'the', '"', 'and', 'to', 'of', 'he', "'", 'a']


In [4]:
text_length=len(text)
num_unique_words=len(words)
print(f"the text contains {text_length} words")
print(f"there are {num_unique_words} unique tokens")  
word_to_int={v:k for k,v in enumerate(words)} 
int_to_word={k:v for k,v in enumerate(words)}
print({k:v for k,v in word_to_int.items() if k in words[:10]})
print({k:v for k,v in int_to_word.items() if v in words[:10]})

the text contains 437112 words
there are 12781 unique tokens
{',': 0, '.': 1, 'the': 2, '"': 3, 'and': 4, 'to': 5, 'of': 6, 'he': 7, "'": 8, 'a': 9}
{0: ',', 1: '.', 2: 'the', 3: '"', 4: 'and', 5: 'to', 6: 'of', 7: 'he', 8: "'", 9: 'a'}


In [5]:
print(text[0:20])
wordidx=[word_to_int[w] for w in text]  
print([word_to_int[w] for w in text[0:20]])  

['chapter', '1', 'happy', 'families', 'are', 'all', 'alike', ';', 'every', 'unhappy', 'family', 'is', 'unhappy', 'in', 'its', 'own', 'way', '.', 'everything', 'was']
[208, 2755, 280, 2981, 83, 31, 2419, 35, 202, 685, 362, 38, 685, 10, 236, 147, 166, 1, 149, 12]


Create batches of training data

In [ ]:
import torch
from torch.utils.data import TensorDataset

seq_len=100  

class TextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data          # data 是整数列表（wordidx）
        self.seq_len = seq_len

    def __len__(self):
        # 总样本数：最后一个有效的起始索引
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # 直接在原始列表上切片（返回 Python list）
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        # 在 DataLoader 内部会自动转为 Tensor，也可以在这里显式转
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


dataset = TextDataset(wordidx, seq_len)


In [ ]:
batch_size = 32
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

x,y=next(iter(loader))

**LSTM Model**

In [ ]:
from torch import nn

device="cuda" if torch.cuda.is_available() else "cpu"

class WordLSTM(nn.Module):
    def __init__(self, input_size=128, n_embed=128,
             n_layers=3, drop_prob=0.2):
        super().__init__()
        self.input_size=input_size
        self.drop_prob = drop_prob
        self.n_layers = n_layers
        self.n_embed = n_embed
        vocab_size=len(word_to_int)
        self.embedding=nn.Embedding(vocab_size,n_embed)
        self.lstm = nn.LSTM(input_size=self.input_size,
            hidden_size=self.n_embed,
            num_layers=self.n_layers,
            dropout=self.drop_prob,batch_first=True)
        self.fc = nn.Linear(self.n_embed, vocab_size)

    def forward(self, x, hc):
        embed=self.embedding(x)
        x, hc = self.lstm(embed, hc)
        x = self.fc(x)
        return x, hc      
          
    def init_hidden(self, n_seqs, device):
    return (torch.zeros(self.n_layers, n_seqs, self.n_embed, device=device),
            torch.zeros(self.n_layers, n_seqs, self.n_embed, device=device))

In [9]:
model=WordLSTM().to(device)

lr=0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_func = nn.CrossEntropyLoss()

In [ ]:
model.train()

for epoch in range(50):
    tloss=0
    sh,sc = model.init_hidden(batch_size)
    for i, (x,y) in enumerate(loader):    
        if x.shape[0]==batch_size:
            inputs, targets = x.to(device), y.to(device)
            optimizer.zero_grad()
            output, (sh,sc) = model(inputs, (sh,sc))
            loss = loss_func(output.transpose(1,2),targets)
            sh,sc=sh.detach(),sc.detach()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            tloss+=loss.item()
        if (i+1)%1000==0:
            print(f"at epoch {epoch} iteration {i+1}\
            average loss = {tloss/(i+1)}")

In [11]:
import os
import pickle

output_dir = r"D:\DeepLearning\DataSet\Model-Weight\LSTM"

model_save_path = os.path.join(output_dir, "wordLSTM.pth")
torch.save(model.state_dict(), model_save_path)

dict_save_path = os.path.join(output_dir, "word_to_int.p")
with open(dict_save_path,"wb") as fb:    
    pickle.dump(word_to_int, fb)

**Generate text with the trained LSTM model**

In [12]:
model.load_state_dict(torch.load(model_save_path))

with open(dict_save_path,"rb") as fb:    
    word_to_int = pickle.load(fb)    
      
int_to_word={v:k for k,v in word_to_int.items()}

In [15]:
import numpy as np

def sample(model, prompt, length=200):
    model.eval()
    text = prompt.lower().split(' ')
    hc = model.init_hidden(1)
    length = length - len(text)
    for i in range(0, length):
        # if the text length is less than seq_len, use text to predict 
        if len(text)<=seq_len:
            x = torch.tensor([[word_to_int[w] for w in text]])
        # otherwise use the last seq_len tokens to predict
        else:
            x = torch.tensor([[word_to_int[w] for w in text[-seq_len:]]])            
        inputs = x.to(device)
        output, hc = model(inputs, hc)
        logits = output[0][-1]
        p = nn.functional.softmax(logits, dim=0).detach().cpu().numpy()
        idx = np.random.choice(len(logits), p=p)
        text.append(int_to_word[idx])
    text=" ".join(text)
    for m in ",.:;?!$()/_&%*@'`":
        text=text.replace(f" {m}", f"{m} ")
    text=text.replace('"  ', '"')   
    text=text.replace("'  ", "'")  
    text=text.replace('" ', '"')   
    text=text.replace("' ", "'")     
    return text  

In [17]:
import torch

torch.manual_seed(42)
np.random.seed(42)
print(sample(model, prompt='Anna and the prince'))  

anna and the prince deferentially untrussing sammt legitimize panic pictured order monogram dissent hollows table lets impropriety frigid oblonskys interrupting peaceful overstrained senselessness pursuit dirt wound rank unworthy rigidity maintenance forgiven disaster delightfully legs shopkeeper dreams working irregularities veiled measuring positive dog exonerate entr lines sublime next nikandrov contented omens checked mute thicker starting irresistibly thrilling cited bonhomie shatter shabby servants experience picture fathers epaulets adultery poetically thronged bass haggle painted spill hot speckly restraint delicious my separates compost surging rearranging evidently denying piece friendships venez police third beds suppositions exasperating cab spats neighboring quick hitch deftly assert rewarded celebrating absurdly blinds together entered veslovsky amalgamated genuinely lovingkindness pillowcases scarlatina districts disorganized carpet beyond esteem uncle flowerbeds deemed 

Temperature and top-K sampling in text generation

In [18]:
def generate(model, prompt, top_k=None, 
             length=200, temperature=1):
    model.eval()
    text = prompt.lower().split(' ')
    hc = model.init_hidden(1)
    length = length - len(text)    
    for i in range(0, length):
        # if the text length is less than seq_len, use text to predict 
        if len(text)<=seq_len:
            x = torch.tensor([[word_to_int[w] for w in text]])
        # otherwise use the last seq_len tokens to predict
        else:
            x = torch.tensor([[word_to_int[w] for w in text[-seq_len:]]])    
        inputs = x.to(device)
        output, hc = model(inputs, hc)
        logits = output[0][-1]
        # scale the logits with the temperature 
        logits = logits/temperature
        p = nn.functional.softmax(logits, dim=0).detach().cpu()    
        if top_k is None:
            idx = np.random.choice(len(logits), p=p.numpy())
        # top-K sampling
        else:
            ps, tops = p.topk(top_k)
            ps=ps/ps.sum()
            idx = np.random.choice(tops, p=ps.numpy())          
        text.append(int_to_word[idx])
    text=" ".join(text)
    for m in ",.:;?!$()/_&%*@'`":
        text=text.replace(f" {m}", f"{m} ")
    text=text.replace('"  ', '"')   
    text=text.replace("'  ", "'")  
    text=text.replace('" ', '"')   
    text=text.replace("' ", "'")     
    return text   

In [19]:
# next token using default setting
prompt="I ' m not going to see"
torch.manual_seed(42)
np.random.seed(42)
for _ in range(10):
    print(generate(model, prompt, top_k=None, 
         length=len(prompt.split(" "))+1, temperature=1)) 

i'm not going to see ridden
i'm not going to see untrussing
i'm not going to see apartments
i'm not going to see saint
i'm not going to see appreciate
i'm not going to see wheat
i'm not going to see marsh
i'm not going to see asthma
i'm not going to see connecting
i'm not going to see borders


In [20]:
# next token using conservative predictions
prompt="I ' m not going to see"
torch.manual_seed(42)
np.random.seed(42)
for _ in range(10):
    print(generate(model, prompt, top_k=3, 
         length=len(prompt.split(" "))+1, temperature=0.5)) 

i'm not going to see breach
i'm not going to see his
i'm not going to see his
i'm not going to see breach
i'm not going to see swollen
i'm not going to see swollen
i'm not going to see swollen
i'm not going to see his
i'm not going to see breach
i'm not going to see his


In [21]:
torch.manual_seed(42)
np.random.seed(42)
print(generate(model, prompt='Anna and the prince',
               top_k=3,
               temperature=0.5)) 

anna and the prince swollen died breach swollen breach breach breach died swollen died swollen died died swollen swollen swollen swollen swollen died,  died, ,  out his swollen,  his his,  swollen, ,  by by his, ,  by his,  his,  by,  his,  his his,  by by by by his by, , , ,  his,  by, ,  his,  by,  by by, ,  by by his by,  his,  by his, , , ,  by his by his,  by by his by his his swollen, , ,  his,  his by,  his by, , , ,  by by his by by,  his his by swollen, , ,  his by by,  his his, , ,  by,  his by his by by,  his, , ,  his his, ,  by, ,  his by,  his by,  by his his his his,  his, , ,  his his,  his,  his,  by his by, , ,  by by,  his by his his, ,  by by his, , 
